In [1]:
!pip install pandas


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [ ]:
import pandas as pd

# 1. Start with sample_unidentified (has demographics + old llm_caseID)
sample = pd.read_csv("../data/sample_unidentified.csv")
print(f"sample_unidentified rows: {len(sample)}")

# 2. Add patient_id from full_data.csv using llm_caseID
full = pd.read_csv("../data/full_data.csv", usecols=["llm_caseID", "patient_id"])
sample = sample.merge(full.rename(columns={"llm_caseID": "llm_caseID_old"}),
                      left_on="llm_caseID", right_on="llm_caseID_old", how="left")
sample = sample.drop(columns=["llm_caseID"])
print(f"After adding patient_id: {sample['patient_id'].notna().sum()} matched")

# 3. Add llm_caseID from processed_data using patient_id
processed = pd.read_csv("../data/processed_data_20260202.csv", usecols=["patient_id", "llm_caseID"])
sample = sample.merge(processed, on="patient_id", how="inner")
print(f"After inner join on patient_id with processed_data: {len(sample)} rows")

# 4. Reorder columns: IDs first
id_cols = ["patient_id", "llm_caseID", "llm_caseID_old"]
other_cols = [c for c in sample.columns if c not in id_cols]
sample = sample[id_cols + other_cols]

print(f"\nFinal rows: {len(sample)}")
print(f"Columns: {list(sample.columns)}")
print(f"\nSample of ID mapping:")
print(sample[["patient_id", "llm_caseID", "llm_caseID_old"]].head(10))

# Save
output_path = "../data/sample_with_demo_patient_id.csv"
sample.to_csv(output_path, index=False)
print(f"\nSaved to {output_path}")

In [ ]:
import pandas as pd

sample = pd.read_csv("../data/sample_with_demo_patient_id.csv", usecols=["patient_id", "llm_caseID"])
processed = pd.read_csv("../data/processed_data_20260202.csv", usecols=["patient_id", "llm_caseID"])

# Merge on patient_id and compare
check = sample.merge(processed, on="patient_id", how="inner", suffixes=("_sample", "_processed"))
mismatches = check[check["llm_caseID_sample"] != check["llm_caseID_processed"]]

print(f"Rows checked: {len(check)}")
print(f"Mismatches:   {len(mismatches)}")

if len(mismatches) == 0:
    print("\n✅ TEST PASSED: llm_caseID matches processed_data for all rows.")
else:
    print("\n❌ TEST FAILED:")
    print(mismatches.head(10))